In [2]:
import fastf1
from fastf1 import Cache

Cache.enable_cache('cache')
Cache.enable_cache(
    cache_dir='cache',
    ignore_version=False,
    force_renew=False,
    use_requests_cache=True
)
schedule = fastf1.get_event_schedule(2025)
schedule = schedule[['RoundNumber', 'EventDate', 'EventName']]
print(schedule)

    RoundNumber  EventDate                  EventName
0             0 2025-02-28         Pre-Season Testing
1             1 2025-03-16      Australian Grand Prix
2             2 2025-03-23         Chinese Grand Prix
3             3 2025-04-06        Japanese Grand Prix
4             4 2025-04-13         Bahrain Grand Prix
5             5 2025-04-20   Saudi Arabian Grand Prix
6             6 2025-05-04           Miami Grand Prix
7             7 2025-05-18  Emilia Romagna Grand Prix
8             8 2025-05-25          Monaco Grand Prix
9             9 2025-06-01         Spanish Grand Prix
10           10 2025-06-15        Canadian Grand Prix
11           11 2025-06-29        Austrian Grand Prix
12           12 2025-07-06         British Grand Prix
13           13 2025-07-27         Belgian Grand Prix
14           14 2025-08-03       Hungarian Grand Prix
15           15 2025-08-31           Dutch Grand Prix
16           16 2025-09-07         Italian Grand Prix
17           17 2025-09-21  

In [3]:
import pandas as pd
import numpy as np


def build_season(year):
    schedule = fastf1.get_event_schedule(year)
    schedule = schedule[schedule['RoundNumber'] > 0]  

    season_rows = []
    for rnd in schedule['RoundNumber']:
        rnd = int(rnd)
        event_name = schedule.loc[schedule['RoundNumber'] == rnd, 'EventName'].values[0]
        try:
            session = fastf1.get_session(year, rnd, 'R')
            qual = fastf1.get_session(year, rnd, 'Q')
            session.load(laps=True, weather=True)
            qual.load(laps=True, weather=True)
        except Exception as e:
            print(f"Skipping {year} round {rnd} ({event_name}): {e}")
            continue

        # Race results already carry starting grid + finishing position
        df = session.results.copy()
        df['Year'] = year
        df['Round'] = rnd
        df['EventName'] = event_name
        df = df.rename(columns={
            'GridPosition': 'StartingGridPosition',
            'Position': 'FinishingPosition',
        })

        # --- Lap time per driver (seconds) ---
        laps = session.laps
        if laps is not None and not laps.empty:
            lap_agg = laps.groupby('Driver')['LapTime'].agg(
                AvgLapTime='mean',
                MedianLapTime='median',
                FastestLapTime='min',
            ).reset_index()
            for col in ['AvgLapTime', 'MedianLapTime', 'FastestLapTime']:
                lap_agg[col] = lap_agg[col].dt.total_seconds()
            df = df.merge(lap_agg, left_on='Abbreviation', right_on='Driver', how='left')
            df = df.drop(columns=['Driver'])

        # --- Weather (session-level; same value for every driver at this track) ---
        weather = session.weather_data
        if weather is not None and not weather.empty:
            df['AirTemp'] = weather['AirTemp'].mean()
            df['TrackTemp'] = weather['TrackTemp'].mean()
            df['Humidity'] = weather['Humidity'].mean()
            df['Pressure'] = weather['Pressure'].mean()
            df['WindSpeed'] = weather['WindSpeed'].mean()
            df['Rainfall'] = bool(weather['Rainfall'].any())


        if qual is not None and not qual.results.empty:
            q = qual.results[['Abbreviation', 'Position', 'Q1', 'Q2', 'Q3']].copy()
            for c in ['Q1', 'Q2', 'Q3']:
                q[c] = q[c].dt.total_seconds()

            best = q[['Q1', 'Q2', 'Q3']].min(axis=1)          # NaT segments collapse away
            q['KnockedOutIn'] = np.where(q['Q3'].notna(), 'Q3',
                                np.where(q['Q2'].notna(), 'Q2', 'Q1'))
            q.loc[best.isna(), 'KnockedOutIn'] = 'NoTime'
            q['QualNoTime'] = best.isna().astype(float)

            gap = 100 * (best - best.min()) / best.min()       # race-relative, not raw seconds
            q['QualGapToPolePct'] = gap.fillna(gap.max() + 1.0)   # no lap -> worse than slowest
            q['QualPosition'] = q['Position'].fillna(20.0)

            df = df.merge(q[['Abbreviation', 'QualPosition', 'QualGapToPolePct',
                             'KnockedOutIn', 'QualNoTime']],
                          on='Abbreviation', how='left', validate='one_to_one')

        season_rows.append(df)

    season_df = pd.concat(season_rows, ignore_index=True)

    keep = [
        'Year', 'Round', 'EventName',
        'Abbreviation', 'DriverNumber', 'TeamName',
        'StartingGridPosition', 'FinishingPosition', 
        'QualPosition', 'QualGapToPolePct', 'KnockedOutIn', 'QualNoTime',
        'AvgLapTime', 'MedianLapTime', 'FastestLapTime',
        'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall',
    ]
    keep = [c for c in keep if c in season_df.columns]
    return season_df[keep]


In [4]:
# Build 2024: weather, lap time, starting grid + finishing position per driver per track
results_2024_df = build_season(2024)
print(results_2024_df.shape)
results_2024_df.head(20)


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Bahrain Grand Prix - Qua

(479, 21)


,Year,Round,EventName,Abbreviation,DriverNumber,TeamName,StartingGridPosition,FinishingPosition,QualPosition,QualGapToPolePct,...,QualNoTime,AvgLapTime,MedianLapTime,FastestLapTime,AirTemp,TrackTemp,Humidity,Pressure,WindSpeed,Rainfall
0,2024,1,Bahrain Grand Prix,VER,1,Red Bull Racing,1.0,1.0,1.0,0.015701,...,0.0,96.574421,95.6790,92.608,18.227389,23.652866,48.821656,1017.185987,0.785987,False
1,2024,1,Bahrain Grand Prix,PER,11,Red Bull Racing,5.0,2.0,5.0,0.417204,...,0.0,96.968404,96.2490,94.364,18.227389,23.652866,48.821656,1017.185987,0.785987,False
2,2024,1,Bahrain Grand Prix,SAI,55,Ferrari,4.0,3.0,4.0,0.383559,...,0.0,97.014947,96.2200,94.507,18.227389,23.652866,48.821656,1017.185987,0.785987,False
3,2024,1,Bahrain Grand Prix,LEC,16,Ferrari,2.0,4.0,2.0,0.000000,...,0.0,97.270368,96.7960,94.090,18.227389,23.652866,48.821656,1017.185987,0.785987,False
4,2024,1,Bahrain Grand Prix,RUS,63,Mercedes,3.0,5.0,3.0,0.358885,...,0.0,97.395263,96.6830,95.065,18.227389,23.652866,48.821656,1017.185987,0.785987,False
5,2024,1,Bahrain Grand Prix,NOR,4,McLaren,7.0,6.0,7.0,0.503561,...,0.0,97.424561,96.6200,94.476,18.227389,23.652866,48.821656,1017.185987,0.785987,False
6,2024,1,Bahrain Grand Prix,HAM,44,Mercedes,9.0,7.0,9.0,0.611226,...,0.0,97.457298,96.6940,94.722,18.227389,23.652866,48.821656,1017.185987,0.785987,False
7,2024,1,Bahrain Grand Prix,PIA,81,McLaren,8.0,8.0,8.0,0.580945,...,0.0,97.558316,96.7960,94.774,18.227389,23.652866,48.821656,1017.185987,0.785987,False
8,2024,1,Bahrain Grand Prix,ALO,14,Aston Martin,6.0,9.0,6.0,0.422812,...,0.0,97.888228,97.2650,94.199,18.227389,23.652866,48.821656,1017.185987,0.785987,False
9,2024,1,Bahrain Grand Prix,STR,18,Aston Martin,12.0,10.0,12.0,0.897213,...,0.0,98.209789,97.1750,95.632,18.227389,23.652866,48.821656,1017.185987,0.785987,False


In [5]:
# Build 2025: weather, lap time, starting grid + finishing position per driver per track
results_2025_df = build_season(2025)
print(results_2025_df.shape)
results_2025_df.head(20)


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INF

(479, 21)


,Year,Round,EventName,Abbreviation,DriverNumber,TeamName,StartingGridPosition,FinishingPosition,QualPosition,QualGapToPolePct,...,QualNoTime,AvgLapTime,MedianLapTime,FastestLapTime,AirTemp,TrackTemp,Humidity,Pressure,WindSpeed,Rainfall
0,2025,1,Australian Grand Prix,NOR,4,McLaren,1.0,1.0,1.0,0.000000,...,0.0,103.428302,90.551,82.167,15.707865,18.942135,78.421348,1009.901685,3.475281,True
1,2025,1,Australian Grand Prix,VER,1,Red Bull Racing,3.0,2.0,3.0,0.512677,...,0.0,103.341151,91.271,83.081,15.707865,18.942135,78.421348,1009.901685,3.475281,True
2,2025,1,Australian Grand Prix,RUS,63,Mercedes,4.0,3.0,4.0,0.599233,...,0.0,103.686340,91.856,85.065,15.707865,18.942135,78.421348,1009.901685,3.475281,True
3,2025,1,Australian Grand Prix,ANT,12,Mercedes,16.0,4.0,16.0,1.902898,...,0.0,104.579370,93.697,84.901,15.707865,18.942135,78.421348,1009.901685,3.475281,True
4,2025,1,Australian Grand Prix,ALB,23,Williams,6.0,5.0,6.0,0.853574,...,0.0,104.672389,93.051,84.597,15.707865,18.942135,78.421348,1009.901685,3.475281,True
5,2025,1,Australian Grand Prix,STR,18,Aston Martin,13.0,6.0,13.0,1.695164,...,0.0,104.730130,93.631,85.538,15.707865,18.942135,78.421348,1009.901685,3.475281,True
6,2025,1,Australian Grand Prix,HUL,27,Kick Sauber,17.0,7.0,17.0,1.974806,...,0.0,104.740167,94.064,85.243,15.707865,18.942135,78.421348,1009.901685,3.475281,True
7,2025,1,Australian Grand Prix,LEC,16,Ferrari,7.0,8.0,7.0,0.877543,...,0.0,103.933528,92.448,85.271,15.707865,18.942135,78.421348,1009.901685,3.475281,True
8,2025,1,Australian Grand Prix,PIA,81,McLaren,2.0,9.0,2.0,0.111857,...,0.0,102.084462,90.722,83.242,15.707865,18.942135,78.421348,1009.901685,3.475281,True
9,2025,1,Australian Grand Prix,HAM,44,Ferrari,8.0,10.0,8.0,1.095931,...,0.0,103.967189,92.914,84.218,15.707865,18.942135,78.421348,1009.901685,3.475281,True


In [6]:
import openpyxl as xl
results_2024_df.to_excel('f1_2024_results.xlsx', index=False)
results_2025_df.to_excel('f1_2025_results.xlsx', index=False)


In [7]:
qual = fastf1.get_session(2025, 1, 'Q')
qual.load(laps=True, weather=True)
display(qual.results.head(20))

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '4'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '1', '63', '22', '23', '16', '44', '10', '55', '6', '14', '18', '7', '5', '12', '27', '30', '31', '87']


,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
4,4,L NORRIS,NOR,norris,McLaren,FF8000,mclaren,Lando,Norris,Lando Norris,...,1.0,,NaN,0 days 00:01:15.912000,0 days 00:01:15.415000,0 days 00:01:15.096000,NaT,,NaN,NaN
81,81,O PIASTRI,PIA,piastri,McLaren,FF8000,mclaren,Oscar,Piastri,Oscar Piastri,...,2.0,,NaN,0 days 00:01:16.062000,0 days 00:01:15.468000,0 days 00:01:15.180000,NaT,,NaN,NaN
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,3671C6,red_bull,Max,Verstappen,Max Verstappen,...,3.0,,NaN,0 days 00:01:16.018000,0 days 00:01:15.565000,0 days 00:01:15.481000,NaT,,NaN,NaN
63,63,G RUSSELL,RUS,russell,Mercedes,27F4D2,mercedes,George,Russell,George Russell,...,4.0,,NaN,0 days 00:01:15.971000,0 days 00:01:15.798000,0 days 00:01:15.546000,NaT,,NaN,NaN
22,22,Y TSUNODA,TSU,tsunoda,Racing Bulls,6692FF,rb,Yuki,Tsunoda,Yuki Tsunoda,...,5.0,,NaN,0 days 00:01:16.225000,0 days 00:01:16.009000,0 days 00:01:15.670000,NaT,,NaN,NaN
23,23,A ALBON,ALB,albon,Williams,64C4FF,williams,Alexander,Albon,Alexander Albon,...,6.0,,NaN,0 days 00:01:16.245000,0 days 00:01:16.017000,0 days 00:01:15.737000,NaT,,NaN,NaN
16,16,C LECLERC,LEC,leclerc,Ferrari,E80020,ferrari,Charles,Leclerc,Charles Leclerc,...,7.0,,NaN,0 days 00:01:16.029000,0 days 00:01:15.827000,0 days 00:01:15.755000,NaT,,NaN,NaN
44,44,L HAMILTON,HAM,hamilton,Ferrari,E80020,ferrari,Lewis,Hamilton,Lewis Hamilton,...,8.0,,NaN,0 days 00:01:16.213000,0 days 00:01:15.919000,0 days 00:01:15.973000,NaT,,NaN,NaN
10,10,P GASLY,GAS,gasly,Alpine,0093CC,alpine,Pierre,Gasly,Pierre Gasly,...,9.0,,NaN,0 days 00:01:16.328000,0 days 00:01:16.112000,0 days 00:01:15.980000,NaT,,NaN,NaN
55,55,C SAINZ,SAI,sainz,Williams,64C4FF,williams,Carlos,Sainz,Carlos Sainz,...,10.0,,NaN,0 days 00:01:16.360000,0 days 00:01:15.931000,0 days 00:01:16.062000,NaT,,NaN,NaN


In [8]:
from sklearn.feature_extraction import DictVectorizer
#now we have all the data we need from the results df. Lets use this to build a vector of features we will train our model on

#lets get an array, where each entry is a dict of our data that we will train on.
#for each race, we want weather, starting grid, finishing grid, track, laptime, driver, team, and year. We can add more in the future
CATEGORICAL = ['EventName', 'Abbreviation', 'TeamName', 'Year', 'KnockedOutIn']
NUMERIC = ['StartingGridPosition', 'QualPosition', 'QualGapToPolePct', 'QualNoTime',
'AvgLapTime', 'MedianLapTime', 'FastestLapTime',
           'AirTemp', 'TrackTemp', 'Humidity', 'Pressure', 'WindSpeed', 'Rainfall']
TARGET = 'FinishingPosition'

def to_records(df):
    out = df.copy()
    for c in CATEGORICAL:
        out[c] = out[c].astype(str)
    for c in NUMERIC:
        out[c] = out[c].astype(float)
    
    records = [
        {k: v for k, v in row.items() if not (isinstance(v, float) and np.isnan(v))}  # drop NaN values
        for row in out[CATEGORICAL + NUMERIC].to_dict(orient='records')
    ]
    y = df[TARGET].to_numpy() if TARGET in df else None
    return records, y

features, Y = to_records(results_2024_df)
vector = DictVectorizer()
X = vector.fit_transform(features).toarray()

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

df = results_2024_df.sort_values('Round').reset_index(drop=True)

train_df = df[df.Round <= 19]
test_df  = df[df.Round > 19]

feat_train, y_train = to_records(train_df)
feat_test,  y_test  = to_records(test_df)

vector = DictVectorizer(sparse=False)
X_train = vector.fit_transform(feat_train)   # fit on train only
X_test  = vector.transform(feat_test)        # transform, never fit

model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)



mae = mean_absolute_error(y_test, pred)
scored = test_df.assign(pred=pred)
rho = np.mean([spearmanr(g[TARGET], g['pred']).statistic
               for _, g in scored.groupby('Round')])


In [ ]:
print(f"feats={X_train.shape[1]}  MAE={mae:.2f}  rho={rho:+.3f}")

scored['PredictedFinish'] = (scored.groupby('Round')['pred']
                                   .rank(method='first').astype(int))

out = (scored.sort_values(['Round', 'FinishingPosition'])
             [['Round', 'EventName', 'Abbreviation', 'StartingGridPosition',
               'FinishingPosition', 'PredictedFinish', 'pred']]
             .rename(columns={'EventName': 'Track', 'Abbreviation': 'Driver',
                              'StartingGridPosition': 'Grid',
                              'FinishingPosition': 'Actual',
                              'PredictedFinish': 'Predicted',
                              'pred': 'RawScore'}))

out['Error'] = out['Predicted'] - out['Actual']

out.to_excel('predictions_2024.xlsx', index=False)

feats=70  MAE=3.21  rho=+0.697
